# W4｜赋值以后，到底改了什么？
在 VS Code 打开本目录并选择 `week-04/.venv` 的 kernel。第一次演示先重启 kernel，随后按实验组运行。
每组独立实验都有自己的初始化；同一组内先修改再观察，不在观察前重新初始化。
本 Notebook 的参考输出用于教师展示与学生复现，不替代 `variant.py` 和自己的测试。
标明 NameError／TypeError 的 cell 是故意的反例，看到预期错误后继续下一组。

## 从 Bash 到 Notebook：代码在哪里执行？
终端的 Python 进程与 Notebook kernel 不同。在新 kernel 查询终端里曾定义的名字，它不会自动出现。下一 cell 故意产生 NameError。

In [1]:
print(probe)

NameError: name 'probe' is not defined

### 初始化 → 第一次观察 → 修改 → 再次观察
分别运行四个 cell。最后指着两次观察问：为什么旧输出仍显示 42？

In [2]:
probe = 42

In [3]:
print(probe)

42


In [4]:
probe = 2026

In [5]:
print(probe)

2026


重新运行前面的“第一次观察”，现在应输出 2026；不要先运行初始化。
手动重启 kernel，再只运行任一观察 cell，应为 NameError。完成后按顺序恢复本组；清空输出不会重置变量。

## 导师要追加实验，旧配置却变了
昨天已用 seed=42 和三组投针数做出论文图表。导师要求保留旧方案，换种子并增加最大投针数。
在修改前预测：source 会不会变化？

In [6]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}

In [7]:
variant = source
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
追加实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}


In [8]:
variant["seed"] = 2026
variant["sample_sizes"][2] = 500000

In [9]:
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}


## 内容相等，是不是同一对象？
保留上一组的 source、variant，再建立内容相等的第三份配置。

In [10]:
same_content = {"seed": 2026, "sample_sizes": [1000, 10000, 500000]}
print(source == variant, source is variant)
print(source == same_content, source is same_content)

True True
True False


## 数字没跟着变：名字重新绑定
下面是新的实验组。先预测：seed 换成 2026，old_seed 会不会一起变？

In [11]:
seed = 42
old_seed = seed
print(seed is old_seed)

True


In [12]:
seed = 2026
print(seed, old_seed, seed is old_seed)

2026 42 False


## 列表跟着变：对象内容被修改
预测 other 与 saved_item 的结果；第三项位置是 2。

In [13]:
l = [1000, 10000, 100000]
other = l
saved_item = l[2]
print(l is other)

True


In [14]:
l[2] = 500000
print(l, other, l is other)
print(saved_item, l[2], saved_item is l[2])

[1000, 10000, 500000] [1000, 10000, 500000] True
100000 500000 False


## 复制以后，先分别改两项
新实验组，先只改 seed，再改第三个投针数。不要在两次修改之间重新初始化。

In [15]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source.copy()
print(source is variant)

False


In [16]:
variant["seed"] = 2026
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 100000]}


In [17]:
variant["sample_sizes"][2] = 500000
print("旧实验：", source)
print("追加实验：", variant)

旧实验： {'seed': 42, 'sample_sizes': [1000, 10000, 500000]}
追加实验： {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}


## 沿关系再查一层
接着上一组，检查字典和列表分别是不是同一对象。

In [18]:
print(source is variant)
print(source["sample_sizes"] is variant["sample_sizes"])

False
True


## “旧配置没变”的检查也可能被骗
下面三段各自初始化。直接引用、浅复制快照和独立基准分别回答什么？

In [19]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
before = source
variant = source
variant["sample_sizes"][2] = 500000
print(source == before)
print(source)

True
{'seed': 42, 'sample_sizes': [1000, 10000, 500000]}


In [20]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
before = source.copy()
variant = source
variant["sample_sizes"][2] = 500000
print(source == before, source["sample_sizes"] is before["sample_sizes"])

True True


In [21]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
expected_source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source
variant["sample_sizes"][2] = 500000
print(source == expected_source)

False


## 怎样让两版配置真正独立？
新字典配新列表；预期两版内容不同，两层身份也不同。

In [22]:
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
variant = source.copy()
variant["sample_sizes"] = source["sample_sizes"].copy()
variant["seed"] = 2026
variant["sample_sizes"][2] = 500000
print(source)
print(variant)
print(source is variant, source["sample_sizes"] is variant["sample_sizes"])

{'seed': 42, 'sample_sizes': [1000, 10000, 100000]}
{'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
False False


## 同参数两次调用：内容相等，仍要各自独立
这里使用 examples.py 的已公开参考，不替你实现 variant.py。
预测 first == second、first is second，以及修改 first 后另两份是否保持。

In [23]:
from examples import reference as make_variant
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
first = make_variant(source, 2026, 2, 500000)
second = make_variant(source, 2026, 2, 500000)
print(first == second, first is second,
      first["sample_sizes"] is second["sample_sizes"])

True False False


In [24]:
first["seed"] = 2028
first["sample_sizes"][1] = 250000
print("first:", first)
print("second:", second)
print("source:", source)
assert second == {"seed": 2026, "sample_sizes": [1000, 10000, 500000]}
assert source == {"seed": 42, "sample_sizes": [1000, 10000, 100000]}

first: {'seed': 2028, 'sample_sizes': [1000, 250000, 500000]}
second: {'seed': 2026, 'sample_sizes': [1000, 10000, 500000]}
source: {'seed': 42, 'sample_sizes': [1000, 10000, 100000]}


## 选做：参数没变化，是否仍需要独立？
先执行直接共享故障，观察它能通过内容检查；再修改结果，查看旧配置。

In [25]:
from examples import direct
source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
expected_source = {"seed": 42, "sample_sizes": [1000, 10000, 100000]}
result = direct(source, 42, 2, 100000)
print("内容正确:", result == expected_source, source == expected_source)
print("身份共享:", result is source,
      result["sample_sizes"] is source["sample_sizes"])

内容正确: True True
身份共享: True True


In [26]:
result["sample_sizes"][0] = 2000
print(source)
print("旧内容保持:", source == expected_source)

{'seed': 42, 'sample_sizes': [2000, 10000, 100000]}
旧内容保持: False


## 备用：字符串能原地改字符吗？
下一 cell 故意产生 TypeError，再运行后面的新实验。

In [27]:
s = "cat"
s[1] = "u"

TypeError: 'str' object does not support item assignment

In [28]:
s = "cat"
old = s
s = s.replace("a", "u")
print(s, old, s is old)

cut cat False


## 备用：相同结果，不同的对象关系
两段各自初始化；先预测，再运行。

In [29]:
a = [1, 2]
b = a
a += [3]
print(a, b, a is b)

[1, 2, 3] [1, 2, 3] True


In [30]:
a = [1, 2]
b = a
a = a + [3]
print(a, b, a is b)

[1, 2, 3] [1, 2] False


In [31]:
a = [1, 2]
b = a
a[0] = 9
a = a + [3]
print(a, b, a is b)

[9, 2, 3] [9, 2] False


## 回到本周作业
独立编写 tests/test_student.py，先让同一份测试接受参考、拒绝 direct/shallow，再实现 variant.py。
说明一条断言的依据及实际共享关系。Notebook 展示不等于你自己的函数和测试已完成。